# Gold Layer — Dynamic CDF-Driven Aggregation

## Architecture
Gold reads the `gold_agg_sql` template from `pipeline_config` and executes it dynamically.
**No code changes are needed to add a new Gold table — add a row to `pipeline_config`.**

## Load Type Strategy
| `gold_load_type` | When to Use | Mechanism |
|---|---|---|
| `full` | Small Silver tables / aggregations that must be 100% accurate | `CREATE OR REPLACE TABLE ... AS SELECT` + OPTIMIZE |
| `incremental` | Large Silver tables where only a subset of keys changed | CDF from Silver → MERGE INTO Gold aggregate |

## SQL Template Variables
The `gold_agg_sql` field in `pipeline_config` supports these placeholders:
- `{silver_<table>}` → resolves to the fully-qualified Silver table name
- `{bronze_<table>}` → resolves to the fully-qualified Bronze table name
- `{gold_<table>}` → resolves to the fully-qualified Gold table name

Example:
```sql
SELECT d.name, d.nationality, COUNT(*) AS number_of_wins
FROM {silver_drivers} d JOIN {silver_results} r ON d.driver_id = r.driver_id
WHERE r.position = 1 GROUP BY d.name, d.nationality
```

In [ ]:
%run ./fw_0.config

In [ ]:
# ── Step 1: Provision Gold schema ─────────────────────────────────────────────

from pyspark.sql import functions as F, Row
from pyspark.sql.window import Window
from datetime import datetime

spark.sql(f"USE CATALOG {CATALOG_NAME}")
spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}
    MANAGED LOCATION '{GOLD_PATH}'
    COMMENT 'Aggregated, business-ready Gold layer'
""")
print(f"Gold schema ready: {CATALOG_NAME}.{GOLD_SCHEMA}")

In [ ]:
# ── Step 2: SQL template resolver ─────────────────────────────────────────────

def resolve_sql_template(sql_template: str, all_configs: list) -> str:
    """
    Replace {silver_<name>}, {bronze_<name>}, {gold_<name>} placeholders
    in gold_agg_sql with fully-qualified Unity Catalog table names.
    """
    resolved = sql_template
    for cfg in all_configs:
        resolved = resolved.replace(
            f"{{silver_{cfg.source_name}}}", fq(SILVER_SCHEMA, cfg.silver_table)
        ).replace(
            f"{{bronze_{cfg.source_name}}}", fq(BRONZE_SCHEMA, cfg.bronze_table)
        )
        if cfg.gold_table:
            resolved = resolved.replace(
                f"{{gold_{cfg.source_name}}}", fq(GOLD_SCHEMA, cfg.gold_table)
            )
    return resolved


print("SQL template resolver loaded.")

In [ ]:
# ── Step 3: Gold watermark table (for incremental load type) ──────────────────

GOLD_WATERMARK_TABLE = fq(FRAMEWORK_SCHEMA, "gold_watermark")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {GOLD_WATERMARK_TABLE} (
        source_name           STRING    NOT NULL,
        last_silver_version   BIGINT    NOT NULL COMMENT 'Last Silver Delta version read by Gold',
        updated_at            TIMESTAMP NOT NULL
    )
    USING DELTA
    TBLPROPERTIES ('quality' = 'framework')
    COMMENT 'CDF watermark — tracks last Silver version processed per Gold table'
""")
print(f"Gold watermark table ready: {GOLD_WATERMARK_TABLE}")


def get_gold_watermark(source_name: str) -> int:
    rows = (
        spark.table(GOLD_WATERMARK_TABLE)
        .filter(F.col("source_name") == source_name)
        .collect()
    )
    return int(rows[0].last_silver_version) if rows else 0


def update_gold_watermark(source_name: str, version: int):
    spark.createDataFrame([
        Row(source_name=source_name, last_silver_version=version, updated_at=datetime.utcnow())
    ]).createOrReplaceTempView("gold_wm_update")
    spark.sql(f"""
        MERGE INTO {GOLD_WATERMARK_TABLE} AS tgt
        USING gold_wm_update AS src
        ON tgt.source_name = src.source_name
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

In [ ]:
# ── Step 4: Core Gold processing function ─────────────────────────────────────

def process_gold(cfg, all_configs: list) -> dict:
    """
    Build or incrementally refresh one Gold table from pipeline_config.
    Skips configs where gold_table or gold_agg_sql is NULL.
    """
    if not cfg.gold_table or not cfg.gold_agg_sql:
        print(f"\n[GOLD] SKIP: {cfg.source_name} — no gold_table or gold_agg_sql defined.")
        return {"status": "SKIPPED", "rows": 0}

    gold_table    = fq(GOLD_SCHEMA, cfg.gold_table)
    silver_table  = fq(SILVER_SCHEMA, cfg.silver_table)
    load_type     = (cfg.gold_load_type or "full").lower()
    resolved_sql  = resolve_sql_template(cfg.gold_agg_sql, all_configs)

    print(f"\n[GOLD] Starting: {cfg.source_name}  →  {gold_table}  (load_type={load_type})")

    if load_type == "full":
        # ── Full recalculation via CREATE OR REPLACE ────────────────────────
        spark.sql(f"""
            CREATE OR REPLACE TABLE {gold_table}
            USING DELTA
            TBLPROPERTIES (
                'quality' = 'gold',
                'delta.enableChangeDataFeed' = 'true'
            )
            AS {resolved_sql}
        """)

    elif load_type == "incremental":
        # ── Incremental via Silver CDF → MERGE INTO Gold ───────────────────
        from_version   = get_gold_watermark(cfg.source_name)
        detail         = spark.sql(f"DESCRIBE HISTORY {silver_table} LIMIT 1").collect()
        latest_version = int(detail[0].version) if detail else 0

        if from_version >= latest_version:
            print(f"         [SKIP] No new Silver versions (watermark={from_version}).")
            return {"status": "SKIPPED", "rows": 0}

        # Re-run aggregation scoped to keys that appear in the Silver CDF batch
        changed_keys = (
            spark.read
            .format("delta")
            .option("readChangeFeed",  "true")
            .option("startingVersion", from_version + 1)
            .table(silver_table)
            .filter(F.col("_change_type").isin("insert", "update_postimage"))
            .select(*[pk.strip() for pk in cfg.primary_key.split(",")])
            .distinct()
        )
        changed_keys.createOrReplaceTempView(f"gold_changed_{cfg.source_name}")

        # Re-aggregate only for the changed keys, then MERGE into Gold
        incremental_sql = f"""
            SELECT agg.*
            FROM ({resolved_sql}) agg
            JOIN gold_changed_{cfg.source_name} ck ON agg.name = ck.name
        """
        spark.sql(incremental_sql).createOrReplaceTempView(f"gold_src_{cfg.source_name}")

        spark.sql(f"""
            MERGE INTO {gold_table} AS tgt
            USING gold_src_{cfg.source_name} AS src
            ON tgt.name = src.name AND tgt.nationality = src.nationality
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)
        update_gold_watermark(cfg.source_name, latest_version)

    # ── Post-build: OPTIMIZE ZORDER ────────────────────────────────────────
    spark.sql(f"OPTIMIZE {gold_table} ZORDER BY (number_of_wins)")

    row_count = spark.table(gold_table).count()
    print(f"[GOLD] Done: {gold_table}  →  {row_count:,} rows")
    return {"status": "SUCCESS", "rows": row_count}


print("Gold processing function loaded.")

In [ ]:
# ── Step 5: Process all active Gold configs ────────────────────────────────────

all_configs = get_pipeline_configs(filter_active=True)
results     = []

for cfg in all_configs:
    try:
        result = process_gold(cfg, all_configs)
        log_pipeline_run(cfg.config_id, "gold", result["status"], result["rows"])
        results.append({"source": cfg.source_name, **result})

    except Exception as e:
        error_msg = str(e)
        print(f"[GOLD] FAILED: {cfg.source_name}  →  {error_msg}")
        log_pipeline_run(cfg.config_id, "gold", "FAILURE", error_msg=error_msg)
        results.append({"source": cfg.source_name, "status": "FAILURE", "error": error_msg})

print("\n" + "="*60)
print("GOLD BUILD SUMMARY")
print("="*60)
for r in results:
    rows = r.get('rows', 'N/A')
    print(f"  {r['source']:20s}  {r.get('status','?'):10s}  {str(rows):>12s} rows")

In [ ]:
# ── Step 6: Data Quality Gate ────────────────────────────────────────────────

failures = [r for r in results if r.get("status") == "FAILURE"]
if failures:
    failed_sources = ", ".join(r["source"] for r in failures)
    raise RuntimeError(
        f"[DQ FAIL] Gold build failed for: {failed_sources}. "
        "Check pipeline_run_log for details."
    )

print("[DQ PASS] All Gold tables built successfully.")